# Document Splitting

This notebook shows how to split a PDF into named logical sections using DocuFlow.

It covers:

- Defining sections with a Pydantic model (recommended for pipelines).
- Defining sections with a `DocumentSection` list (convenient for dynamic names).
- Reading `SplitResult` and `page_map`.
- Deep mode: per-section `confidence` and `evidence`.
- No-overlap mode.
- Custom split rules.
- Processing a page subset.
- Using the split result to feed targeted extraction.

## Setup

Install dependencies:

```bash
uv pip install -e ".[pdf,llm,jupyter]"
```

Set `RUN_LLM=1` and configure your provider API key (e.g. `GEMINI_API_KEY`) to run live LLM cells.
Cells that call the LLM are skipped by default and show their expected output instead.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

from pydantic import BaseModel, Field

try:
    from reportlab.pdfgen import canvas as rl_canvas
    from reportlab.lib.pagesizes import letter
except ImportError as exc:
    raise RuntimeError(
        "Install dev dependencies: uv pip install -e '.[dev,pdf,llm]'"
    ) from exc

from docuflow import split_document_async
from docuflow.splitting import DocumentSection, SplitResult

RUN_LLM = os.getenv("RUN_LLM") == "1"
MODEL = os.getenv("DOCUFLOW_SPLIT_MODEL", "gemini/gemini-2.5-flash")

cwd = Path.cwd()
EXAMPLES_DIR = cwd if cwd.name == "examples" else cwd / "examples"
OUTPUT_DIR = EXAMPLES_DIR / "data" / "splitting_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output dir:", OUTPUT_DIR)

## Build a sample multi-section PDF

We create a five-page PDF with clearly named sections so the LLM can assign them reliably.

In [ ]:
def make_contract_pdf(path: Path) -> None:
    c = rl_canvas.Canvas(str(path), pagesize=letter)
    pages = [
        ("CONTRACT TERMS",
         "This agreement sets out the main terms and conditions between the parties."),
        ("CONTRACT TERMS — CONTINUED",
         "Article 5: Liability. The total liability of either party shall not exceed..."),
        ("EXHIBIT A — SPECIFICATIONS",
         "Technical specifications and deliverables attached hereto as Exhibit A."),
        ("EXHIBIT B — PRICING SCHEDULE",
         "Pricing schedule and payment milestones as agreed by both parties."),
        ("SIGNATURE PAGE",
         "IN WITNESS WHEREOF the parties have executed this Agreement.\n\n"
         "Signed: ____________________  Date: ____________"),
    ]
    for heading, body in pages:
        c.setFont("Helvetica-Bold", 14)
        c.drawString(72, 720, heading)
        c.setFont("Helvetica", 11)
        c.drawString(72, 690, body[:90])
        c.showPage()
    c.save()


contract_pdf = OUTPUT_DIR / "sample_contract.pdf"
make_contract_pdf(contract_pdf)
print("Created:", contract_pdf)

## 1. Define sections with a Pydantic model

This is the recommended approach for code-defined pipelines. Field names become section
identifiers; `Field(description=...)` is the natural-language hint sent to the LLM.

In [ ]:
class ContractSections(BaseModel):
    contract_body: str = Field(description="Main contract terms and conditions")
    exhibits:      str = Field(description="Attached exhibits and appendices")
    signature_page: str = Field(description="Pages containing signature blocks")


if not RUN_LLM:
    print(
        "Skipped — set RUN_LLM=1 and configure your provider API key to run LLM cells.\n"
        "Expected output: contract_body=[0,1], exhibits=[2,3], signature_page=[4]"
    )
else:
    result: SplitResult = await split_document_async(
        str(contract_pdf),
        ContractSections,
        model=MODEL,
    )
    print("success:    ", result.success)
    print("total_pages:", result.total_pages)
    print("page_map:   ", result.page_map)
    print("warnings:   ", result.warnings)

## 2. Define sections with a DocumentSection list

Use this form when section names are dynamic or come from user input.

In [ ]:
dynamic_sections = [
    DocumentSection(name="contract_body", description="Main contract terms and conditions"),
    DocumentSection(name="exhibits",      description="Attached exhibits and appendices"),
    DocumentSection(name="signature_page", description="Pages containing signature blocks"),
]

if not RUN_LLM:
    print("Skipped — same result as the Pydantic model approach.")
else:
    result2 = await split_document_async(
        str(contract_pdf), dynamic_sections, model=MODEL
    )
    print("page_map:", result2.page_map)

## 3. Deep mode — confidence and evidence

Pass `deep=True` to ask the LLM to also return a `confidence` level (`"high"` / `"medium"` / `"low"`) and a one-sentence `evidence` explanation per section.

In [ ]:
if not RUN_LLM:
    print(
        "Skipped — set RUN_LLM=1 to run.\n"
        "Expected: each section has confidence='high' and a short evidence string."
    )
else:
    deep_result = await split_document_async(
        str(contract_pdf), ContractSections, model=MODEL, deep=True
    )
    for name, section in deep_result.sections.items():
        print(f"{name}")
        print(f"  pages:      {section.pages}")
        print(f"  confidence: {section.confidence}")
        print(f"  evidence:   {section.evidence}")
        print()

## 4. No-overlap mode

By default a page may appear in more than one section (e.g. a transition page that
belongs to both the contract body and the first exhibit). Pass `allow_overlap=False`
to enforce exclusive assignment — each page goes to at most one section.

In [ ]:
if not RUN_LLM:
    print("Skipped — set RUN_LLM=1 to run.")
else:
    no_overlap = await split_document_async(
        str(contract_pdf), ContractSections, model=MODEL, allow_overlap=False
    )
    all_pages = [p for pages in no_overlap.page_map.values() for p in pages]
    print("page_map:      ", no_overlap.page_map)
    print("unique pages:  ", sorted(set(all_pages)))
    print("no duplicates?:", len(all_pages) == len(set(all_pages)))

## 5. Custom split rules

Override the LLM's default splitting logic with a freeform instruction. Useful when
your document follows a specific convention the model might not infer on its own.

In [ ]:
CUSTOM_RULES = (
    "Contract body pages start with 'CONTRACT TERMS'. "
    "Exhibit pages start with 'EXHIBIT'. "
    "The signature page contains the word 'SIGNATURE'."
)

if not RUN_LLM:
    print("Skipped — set RUN_LLM=1 to run.")
else:
    custom_result = await split_document_async(
        str(contract_pdf), ContractSections, model=MODEL, split_rules=CUSTOM_RULES
    )
    print("page_map:", custom_result.page_map)

## 6. Processing a page subset

For large documents where you already know the region of interest, pass a list of
0-based page indices. The returned indices are still the original document page numbers.

In [ ]:
# Only send pages 2-4 (exhibits + signature page) to the LLM.
if not RUN_LLM:
    print("Skipped — set RUN_LLM=1 to run.")
else:
    subset_result = await split_document_async(
        str(contract_pdf), ContractSections, model=MODEL, pages=[2, 3, 4]
    )
    print("page_map (subset):", subset_result.page_map)

## 7. Using split results in a pipeline

`result.page_map` returns a plain `dict[str, list[int]]` — feed it directly to
`extract()` or any other DocuFlow function that accepts a page list.

In [ ]:
# Pseudocode — replace split_result with an actual SplitResult from the LLM cells above.
# from docuflow import extract
#
# split_result = await split_document_async("contract.pdf", ContractSections)
# exhibit_pages = split_result.page_map.get("exhibits", [])
#
# extraction = await extract(
#     "contract.pdf",
#     schema=ExhibitSchema,
#     pages=exhibit_pages,
# )

# Demonstrate the page_map property on a mock result:
from docuflow.splitting.models import SectionResult

mock_result = SplitResult(
    input_path="contract.pdf",
    total_pages=5,
    sections={
        "contract_body": SectionResult(pages=[0, 1]),
        "exhibits":      SectionResult(pages=[2, 3]),
        "signature_page": SectionResult(pages=[4]),
    },
)

print("page_map:      ", mock_result.page_map)
print("exhibit pages: ", mock_result.page_map.get("exhibits", []))
print("success:       ", mock_result.success)

## SplitResult Reference

| Attribute | Type | Description |
| --- | --- | --- |
| `result.success` | `bool` | `True` when no errors and at least one section returned. |
| `result.input_path` | `str` | Original file path. |
| `result.total_pages` | `int` | Number of pages processed. |
| `result.model` | `str` | Model used. |
| `result.page_map` | `dict[str, list[int]]` | Section → sorted page indices (property). |
| `result.sections` | `dict[str, SectionResult]` | Full detail per section. |
| `result.sections[name].pages` | `list[int]` | Pages assigned to this section. |
| `result.sections[name].confidence` | `str` | `"high"` / `"medium"` / `"low"` (deep mode). |
| `result.sections[name].evidence` | `str` | LLM explanation (deep mode). |
| `result.usage` | `dict` | `{"prompt_tokens": ..., "completion_tokens": ..., "cost_usd": ...}` |
| `result.warnings` | `list[str]` | Out-of-range pages stripped, etc. |
| `result.errors` | `list[str]` | Empty on success. |